# 1. Orientation: what CIRRUS is

← [Introduction](00-introduction.ipynb) · next: [Containers](02-containers.ipynb)

[Page 0](00-introduction.ipynb) ended on Kubernetes: it keeps a workload up when
servers fail, and its one real cost is that *somebody has to run the cluster*.
CIRRUS is NCAR's answer to that cost. This page is what CIRRUS is, where it runs,
when it is the right answer, how you get on it, and what comes with access —
before any of the tooling. Nothing here is hard, and skipping it is the reason
people spend a week deploying something to the wrong system.

---

> **Before you start.** Three things to know:
>
> 1. **Sign in from a terminal first.** Run `kubectl get pods` in a terminal and
>    complete the device-code prompt. A credential prompt has nowhere to appear
>    inside a kernel, so the first `kubectl` in a notebook will simply time out.
>    You will need to do this again roughly hourly, when the token expires.
> 2. **Run the setup cell below**, once. It sets the working directory and `$IMG`
>    in the kernel, which every `%%bash` cell inherits.
> 3. **A few steps are terminal-only** and appear as plain code blocks rather
>    than runnable cells -- an interactive shell, an endless reconcile loop, two
>    things watched at once. They are marked where they appear.

In [ ]:
# Setup for this notebook -- run this first.
#
# Each %%bash cell below is its own subshell, so a variable set in one
# cell is gone in the next. Setting them here instead puts them in the
# kernel's own environment and working directory, which every %%bash
# subprocess inherits.
import os, pathlib, subprocess

workdir = pathlib.Path(os.environ.get('CIRRUS_WORKDIR',
                                      pathlib.Path.home() / 'cirrus-intro'))
os.chdir(workdir)
print('working directory:', os.getcwd())

## What CIRRUS is

**CIRRUS is NCAR's Kubernetes platform** — *Cloud Infrastructure for Remote
Research, Universities, and Scientists* — run by CISL for long running workloads.

A certain kind of research output is a *service* rather than a job: a dashboard 
over model output, an API in front of a dataset, a documentation site, a 
data-download endpoint, a JupyterHub for a project team. Those things have to be 
up at 3 a.m. on a Sunday, they have to come back on their own when a server 
reboots, and they need a URL and a certificate. A scheduler is the wrong tool 
for those workloads and that is the gap CIRRUS fills by utilizing Kubernetes to 
run containers at scale.

What people actually run on it today is public, and worth browsing before 
designing anything: <https://cirrus.k8s.ucar.edu/apps>.

CIRRUS is standard Kubernetes and uses standard Helm. That means what you build 
here also runs on any other Kubernetes — a cloud provider, a colleague's cluster, 
a laptop with `kind`. That portability is deliberate and it is the strongest 
argument for learning the platform rather than a bespoke deployment process.

---

## Kubernetes, in one screen

[Page 0](00-introduction.ipynb) was *why* Kubernetes exists. This is *what* it is
made of — a handful of words the rest of this page leans on.
[Page 3](03-kubernetes.ipynb) covers them properly; this is just enough to read
what follows.

**A node is one machine.** A physical or virtual server with CPU, memory, maybe a
GPU, and a container runtime. On its own it is just a computer that can run
containers.

**A cluster is many nodes run as one.** Kubernetes joins a set of nodes together
and puts a **control plane** in charge of them — the API server you talk to with
`kubectl`, plus the components that decide what runs where. You stop thinking
about individual machines and start thinking about the cluster's total capacity.

**A pod is the unit it runs.** One container (occasionally a few that belong
together) with a name, an IP address and a resource request — "this needs 2
CPUs and 4 GB of memory". You almost never place a pod yourself.

**The scheduler places pods, and keeps placing them.** When a pod needs a home,
the control plane picks a node with room for it. If that node fails, is drained
for maintenance, or runs short of resources, the pod is recreated on another one.
Pods are expected to come and go; the *service* they provide is what stays up.

```
                 ┌──────────────── cluster ─────────────────┐
  kubectl ──▶    │  control plane  (API server, scheduler)  │
                 │        │             │             │     │
                 │     ┌──▼───┐      ┌──▼───┐      ┌──▼───┐ │
                 │     │node 1│      │node 2│      │node 3│ │
                 │     │ pod  │      │ pod  │      │ pod  │ │
                 │     │ pod  │      │ pod  │      │      │ │
                 │     └──────┘      └──────┘      └──────┘ │
                 └──────────────────────────────────────────┘
```

**You describe what you want, not how to do it.** Instead of "start this
container on that server", you hand the API server a description — "three copies
of this image, reachable at this hostname" — and Kubernetes works continuously to
make it true. That is where the self-healing comes from, and it is the idea
[page 3](03-kubernetes.ipynb#the-one-idea) is built around.

**A namespace is a named slice of the cluster.** Many teams share the same nodes;
namespaces keep their names, permissions and quotas apart. If you have used HPC,
think of it as your allocation. More on those
[below](#your-space-on-the-cluster-namespaces).

That is the whole mental model for now: many machines, one API, and a control
plane that decides where your containers run and moves them when it has to.

---

## Kubernetes, at two sites

CIRRUS is not one cluster. It is a **cluster at each of NCAR's two data centers**, 
which is what makes "the site is down" survivable:

| cluster | site | Argo CD |
| --- | --- | --- |
| **`mlc1`** | Mesa Lab (ML), Boulder | <https://mlc1-argo.k8s.ucar.edu/> |
| **`nwc1`** | NWSC, Cheyenne | <https://nwc1-argo.k8s.ucar.edu/> |

Roughly 18 nodes between them — AMD EPYC CPU nodes plus NVIDIA GPU nodes (A10 at
ML, A2 at NWSC), on the order of 800 CPU cores, 9 TB of memory and 240 TB of
NVMe, on a 25 Gb interconnect. Object storage is replicated *between* the two
sites, so an S3 bucket survives losing one of them
([page 7](07-storage.ipynb)).

Your session is on `mlc1`. Ask, rather than assume:

In [ ]:
%%bash
kubectl config current-context
kubectl version

The control plane is on a recent 1.3x minor and is upgraded roughly quarterly.
The `kubectl` in this image tracks that minor and never leads it, which is why
`cirrus-versions` pins it rather than fetching `stable`.

Two practical consequences of there being two clusters:

* **A deployment lives on one of them.** Running in both is a choice you make
  explicitly — two Argo CD Applications, one hostname in front — not something
  that happens for free.
* **Which one you get is a conversation.** If your application needs to be near
  data that only exists at one site, say so when you ask for it.

---

## When CIRRUS, and when Casper or Derecho

This is the single most useful thing on the page. CIRRUS is a *complement* to
the HPC systems, not a replacement, and the dividing line is sharp: **HPC runs
jobs that finish; CIRRUS runs services that do not.**

| you want to | use |
| --- | --- |
| serve a dashboard, API, website or data portal | **CIRRUS** |
| run something that must stay up, restart itself, survive a node failing | **CIRRUS** |
| deploy from git, continuously, with review | **CIRRUS** |
| host a JupyterHub or a notebook service for a team | **CIRRUS** |
| run CI that needs more than GitHub's free runners, or a GPU | **CIRRUS** ([page 8](08-github-actions.ipynb)) |
| run a coupled model, or MPI across many nodes at scale | **Derecho** |
| run a big single-node analysis, or hold a GPU for hours | **Casper** |
| push a queue of independent batch jobs through | **Casper / Derecho**, via PBS |

The overlap is real but small. There *is* an MPI operator on CIRRUS
([page 13](13-mpi.ipynb)) — it exists for multi-node work that belongs next to
a service or a CI pipeline, not to compete with Derecho's interconnect. If your
question is "how many nodes for how many hours", you want the scheduler.

The combination is the common case, and it is worth designing for on purpose:
**run the model on Derecho, serve the results from CIRRUS.** A CIRRUS
application can be given read-only access to GLADE on request
([page 7](07-storage.ipynb#glade-and-why-your-session-can-see-it-but-your-deployment-cannot)),
so the handoff can be a path rather than a copy.

---

## Getting access

There is no self-service signup, and that is by design — every namespace on the
cluster has a person attached to it.

**Everything starts with a ticket.** The CIRRUS team works from a Jira Kanban
board, and a ticket is how requests get prioritised, tracked and answered.

* New service, application, or access → the **New Service Request** form
* Something broken → the **Report Issue** form
* Both are linked from <https://cirrus.k8s.ucar.edu/> under *Report Issue* and
  *Getting Started*, and there is a shorter application form at
  <https://cirrus.k8s.ucar.edu/request-app>
* Or email <cirrus-admin@ucar.edu>

Fill in **Reporter with your own name** — it is not filled automatically, and a
ticket with no reporter is a ticket nobody can follow up with. Leave *Assignee*
on `Automatic` and *Epic Link* on `User Requests`.

Some things you get without asking, because they follow your UCAR CIT account:

| already yours | not yet yours |
| --- | --- |
| this workshop session | a namespace of your own |
| JupyterHub (<https://jupyter.k8s.ucar.edu/>) and public Binder | `kubectl` access to the cluster from your own machine ([below](#kubectl-from-your-own-machine)) |
| signing in to the image registry, and pulling public images | somewhere in the registry to push your own images ([page 2](02-containers.ipynb)) |
| the secret store, under your own email address | secrets delivered into your namespace automatically ([page 6](06-secrets.ipynb)) |
| S3 keys and buckets, self-service at <https://cs3.k8s.ucar.edu/> | an application deployed from git ([page 5](05-argocd.ipynb)) |
| | CI runners on CIRRUS for your repository ([page 8](08-github-actions.ipynb)) |
| | Grafana, to see your application's logs and metrics ([page 9](09-observability.ipynb)) |
| | the LLM service — sign in once at <https://llm.k8s.ucar.edu/> to join the approval list ([page 14](14-llm.ipynb)) |

The right-hand column is one ticket each, and they are short conversations — the
LLM service is the exception, where signing in *is* the request. You do not need
to know what any of them involve yet: each is explained on the page named next
to it, in the order you would need them. What to include when you ask
for an application to be hosted is on
[page 5](05-argocd.ipynb#onboarding-an-application-on-cirrus).

### `kubectl` from your own machine

Inside this workshop session, `kubectl` is already installed and pointed at your
namespace. On your own laptop or workstation it takes four steps, and you only
need it if you want to drive the cluster yourself — watching pods, reading logs,
or managing things that exist only through the API, like functions and MPI jobs.
An application deployed through Argo CD needs none of this.

1. **Ask for Kubernetes API access** with a ticket. The CIRRUS team creates a
   *tenant* scoped to your namespaces; this is what actually lets you in.
2. **Install [`kubectl`](https://kubernetes.io/docs/tasks/tools/#kubectl) and the
   [Azure `kubelogin`](https://azure.github.io/kubelogin/install.html) plugin.**
   `kubelogin` is what turns a `kubectl` command into a UCAR sign-in.
3. **Download the shared kubeconfig**, from the UCAR network or VPN — off it, the
   download just hangs. Use `curl` rather than a browser, which would display the
   file instead of saving it:

   ```bash
   curl -o ~/.kube/config https://s3.k8s.ucar.edu:5443/cirrus-config/kubeconfig
   ```

   If `~/.kube/config` already holds another cluster, save it elsewhere and
   point at it instead: `curl -o ~/.kube/cirrus-config ...` then
   `export KUBECONFIG=~/.kube/cirrus-config`.
4. **Run any `kubectl` command.** The first one opens a browser sign-in through
   Microsoft Entra ID, with two-factor authentication.

Everyone uses the same kubeconfig; what you can do after signing in depends
entirely on step 1. Before your tenant exists, sign-in succeeds and every request
after it is refused — which looks like a broken setup and is not.

The session you are in does the same thing with one difference: there is no
browser in a container, so it uses the device-code flow instead.
[Page 3](03-kubernetes.ipynb#who-you-are-to-the-cluster) explains the sign-in in
detail. The CIRRUS documentation's
[kubectl page](https://ncar-hpc-docs.readthedocs.io/en/latest/compute-systems/cirrus/guides/10-kubectl/kubectl/)
has tips worth taking on day one: a `k` alias with completion, `kubectx` and
`kubens` for switching, and `kube-ps1` so your prompt always shows where you are.

---

## Your space on the cluster: namespaces

A **namespace** is Kubernetes' unit of ownership: a scope for names, a boundary
for policy, and the thing a quota is attached to. If you have used HPC, it plays
the part of an allocation: your area of the cluster, with access for you and your
team and limits on how much you can use. (Under the hood, a tool called
**Capsule** enforces those boundaries between teams.)

Three shapes, and knowing which one you are asking for saves a round trip:

**An individual namespace.** Yours, named after you. This session has one:

In [ ]:
%%bash
echo "$CIRRUS_NAMESPACE"
kubectl get all

It is where you learn, prototype, and run `kubectl apply` against something real
without a review cycle. It is not where a service that other people depend on
should live.

**A team or project namespace.** Carved out and dedicated to one application or
one group, with several people as owners and its own quota. This is what a hosted
application gets, and it is what you should ask for once something has users.
Because ownership is per-namespace, a colleague added to it can debug your
deployment without being given access to anything else.

**A shared platform namespace.** Where the platform's own components live —
`argocd`, the ingress controllers, the monitoring stack. You will read about
these and you will not be in them.

The fence around your namespace is real, and it is better measured than guessed
at. [Page 3](03-kubernetes.ipynb#your-namespace-and-what-you-may-do-in-it) does that
with `kubectl auth can-i`, quotas and limit ranges, in the context of the objects
they apply to.

---

## What else comes with access

CIRRUS is not only "somewhere to run a container". These platform services come
with it, listed in the order the workshop reaches them. Each pairs with a step on
[page 0](00-introduction.ipynb)'s road:

| service | what it is for | where |
| --- | --- | --- |
| **Harbor** | the container registry: the library your images live in, with security scanning | <https://hub.k8s.ucar.edu/> · [page 2](02-containers.ipynb#container-registries) |
| **Argo CD** | GitOps: the cluster continuously pulls what should be running from your repository | ML / NWSC, above · [page 5](05-argocd.ipynb) |
| **OpenBao** | encrypted secret storage, delivered to your application without ever touching git | <https://bao.k8s.ucar.edu/> · [page 6](06-secrets.ipynb) |
| **Ceph and S3** | storage that outlives a container: volumes, and object storage replicated between sites | [page 7](07-storage.ipynb) |
| **CloudNativePG** | a managed PostgreSQL database, with its credentials from OpenBao | [page 7](07-storage.ipynb#a-database-cloudnativepg) |
| **GitHub Actions runners** | automated tests and image builds on cluster hardware, GPUs included | [page 8](08-github-actions.ipynb) |
| **Argo Workflows** | multi-step and scheduled jobs that run as pods in your namespace | [page 8](08-github-actions.ipynb#the-cluster-native-alternative-argo-workflows) |
| **Grafana** | metrics, logs, dashboards and alerts for what you deployed | <https://grafana.k8s.ucar.edu/> · [page 9](09-observability.ipynb) |
| **JupyterHub and Binder** | notebooks with GPUs, GLADE and Dask, no install | <https://jupyter.k8s.ucar.edu/> · [page 11](11-jupyter.ipynb) |
| **Fission, MPI, the LLM service** | functions on demand, multi-node jobs, a local LLM | [pages 12–14](10-workloads.ipynb) |

And three more things the platform does on your behalf, which is most of why
hosting an application here is worth the ticket:

* **DNS.** ExternalDNS creates the record for your hostname under
  `*.k8s.ucar.edu`. You pick the name; you do not file a DNS request.
* **TLS.** cert-manager issues and renews the certificate. Nobody has a calendar
  reminder to replace it.
* **Ingress.** Two paths — one for the public internet and one restricted to the
  UCAR network — and choosing between them is one value in your chart, not an
  infrastructure project. ([Page 3](03-kubernetes.ipynb#ingress) has the object.)

CIRRUS is free to UCAR employees and collaborators. There is no cloud bill and
no credit card, which is a real advantage and also the reason quotas exist: the
constraint is a shared cluster rather than a budget.

### What the platform promises, and what stays yours

Worth reading once, because it sets expectations correctly. The full service
level agreement is at <https://cirrus.k8s.ucar.edu/sla>; the short version:

* **The team runs the platform.** Cluster availability, upgrades, the monitoring
  stack, backups of Argo CD projects and Harbor images.
* **You run your application.** Its code, its resource requests, its ability to
  survive a pod being moved. "Application owners are responsible for their
  applications" is the actual wording, and the design consequence is that
  **anything that cannot tolerate being restarted does not belong here** — write
  workloads that can be killed at any moment, because they will be.
* **Support is business hours**, Monday to Friday, 08:00–17:00 MST, with no
  after-hours on-call. Critical means a 2-hour response in that window.
* **Backups are the repository.** Applications are recovered by redeploying from
  git, not by restoring a machine. That is another argument for
  [page 5](05-argocd.ipynb), and it means a PersistentVolume holding something
  irreplaceable needs an explicit conversation — replication across sites is
  available on request, not by default.

---

## The road from here

The remaining pages follow one small web application from your laptop to a
hosted service, in the order the work actually happens. It is the same path as
the getting-started steps on [page 0](00-introduction.ipynb), and each page picks
up exactly where the previous one leaves the application:

| page | the step | where it leaves the application |
| --- | --- | --- |
| **[2. Containers](02-containers.ipynb)** | package it as an image, and put the image in a registry | portable and reproducible, but nothing runs it or restarts it |
| **[3. Kubernetes](03-kubernetes.ipynb)** | describe how it should run, and apply that by hand | running and self-healing, as a handful of YAML files |
| **[4. Helm](04-helm.ipynb)** | bundle those files into one configurable chart | installable as one thing, but only when someone runs a command |
| **[5. Argo CD](05-argocd.ipynb)** | put the chart in git and let the cluster follow it | deployed from git, reviewed and recorded |
| **[6. Secrets](06-secrets.ipynb)** | give it credentials without putting them in git | can talk to the things it needs |
| **[7. Storage](07-storage.ipynb)** | give it data that outlives its pods | keeps what it writes |
| **[8. GitHub Actions](08-github-actions.ipynb)** | build and test the image automatically on every change | a merge is all it takes to ship |
| **[9. Observability](09-observability.ipynb)** | collect its logs and metrics, and alert on them | you hear about problems before users do |
| **[10–14. Specialized workloads](10-workloads.ipynb)** | notebooks, functions, MPI and LLMs: work that is not a long-running web service | — |

Pages 2 through 4 are true of any Kubernetes cluster. From page 5 on, it is
increasingly about *this* one.

---

## Check yourself

1. You have a model that runs for six hours on 40 nodes, and a dashboard that
   shows its output. Which goes where, and why?
2. CIRRUS has clusters at two sites. What does that get you for free, and what
   does it not?
3. Name two things you already have access to today, and two that need a ticket.
4. Your application must not lose data if a pod is killed without warning. What
   does the SLA imply you have to do about that?
5. Where in the road from here does your application first stop depending on
   someone typing a command?

---

← [Introduction](00-introduction.ipynb) · next: [2. Containers](02-containers.ipynb)